<a href="https://colab.research.google.com/github/nisaa783/scikit-learn-cookbook_Anisa-Khasanah/blob/main/Chapter_1_Common_Conventions_and_API_Elements_of_scikit_learn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Pendahuluan dan Filosofi Desain API scikit-learn**
Scikit-learn dibangun di atas prinsip konsistensi, kemudahan penggunaan, dan integrasi yang erat dengan pustaka komputasi numerik Python seperti NumPy dan SciPy. Filosofi utama dari perpustakaan ini adalah keseragaman antarmuka (interface). Apa pun algoritma machine learning yang Anda gunakan—apakah itu regresi linear, support vector machine, ataupun random forest—cara Anda memanggil fungsi untuk melatih model, melakukan prediksi, atau mengubah data akan selalu mengikuti pola yang sama persis.

Keseragaman ini sangat menghemat waktu pengembang karena Anda tidak perlu mempelajari ulang sintaks khusus untuk setiap algoritma yang berbeda. Selain itu, scikit-learn merancang objek-objektnya agar dapat dikomposisikan secara fleksibel, terutama melalui konsep Estimator dan Pipeline.

## **2. Tiga Pilar Utama Objek scikit-learn**
Dalam ekosistem scikit-learn, hampir semua objek dikategorikan ke dalam tiga peran utama berdasarkan fungsionalitasnya:

Estimator: Objek apa pun yang dapat mempelajari (mengekstrak pola dari) data berdasarkan parameter yang diberikan. Estimator mencakup algoritma pembelajaran mesin maupun fungsi utilitas preprocessing. Estimator diinisialisasi sebagai objek kelas, kemudian dilatih menggunakan metode .fit(X, y) di mana X adalah matriks fitur dan y adalah label target (untuk supervised learning).

Transformer: Jenis khusus dari estimator yang memiliki kemampuan untuk mengubah atau memanipulasi data masukan (X) menjadi format baru (misalnya normalisasi, penskalaan, atau reduksi dimensi). Transformer menggunakan metode .transform(X) atau digabung menjadi .fit_transform(X).

Predictor: Estimator yang memiliki kemampuan tambahan untuk menghasilkan prediksi ketika diberikan data baru yang belum pernah dilihat model sebelumnya. Predictor menggunakan metode .predict(X) untuk klasifikasi/regresi, serta memiliki metode .score(X, y) untuk mengevaluasi tingkat akurasi atau performa model.

## **3. Otomasi Alur Kerja dengan Pipeline**
Salah satu kesalahan pemula yang paling sering terjadi dalam machine learning adalah melakukan penskalaan data (scaling) atau imputasi pada keseluruhan dataset sebelum membaginya menjadi data latih dan data uji (train-test split). Hal ini memicu masalah serius yang disebut data leakage (kebocoran data), di mana informasi dari data uji "bocor" ke dalam proses pelatihan model, membuat performa model terlihat sangat bagus secara artifisial tetapi gagal total di dunia nyata.

Scikit-learn mengatasi masalah ini melalui Pipeline. Pipeline memungkinkan kita menggabungkan beberapa langkah transformer dan satu estimator akhir ke dalam satu objek terpadu. Ketika metode .fit() dipanggil pada pipeline, proses fitting transformer hanya dilakukan berdasarkan data pelatihan di dalam setiap lipatan validasi silang, sehingga kebersihan data terjaga secara sempurna.

## **4. Contoh Program Penerapan 1: Dasar Penggunaan Estimator, **Transformer, dan Predictor secara Manual
Pada contoh pertama ini, kita akan melihat bagaimana tahapan dasar dalam scikit-learn dijalankan secara terpisah—mulai dari memuat data, menskalakan fitur menggunakan transformer (StandardScaler), hingga melatih dan memprediksi menggunakan estimator/predictor (LogisticRegression).



In [1]:
# Import pustaka yang diperlukan
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Memuat dataset contoh (Iris Dataset)
data = load_iris()
X = data.data  # Matriks fitur (150 sampel, 4 fitur)
Y = data.target  # Vektor target/label kelas (0, 1, atau 2)

# Membagi data menjadi data latih (70%) dan data uji (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.3, random_state=42
)

print(f"Dimensi X_train: {X_train.shape}")
print(f"Dimensi X_test: {X_test.shape}")

# 2. Menerapkan Transformer (StandardScaler)
# Inisialisasi transformer untuk menyeragamkan skala fitur (mean=0, variance=1)
scaler = StandardScaler()

# Melatih transformer HANYA pada data latih dan mentransformasikannya
X_train_scaled = scaler.fit_transform(X_train)

# Menerapkan parameter transformer yang sama ke data uji (mencegah data leakage)
X_test_scaled = scaler.transform(X_test)

# 3. Menerapkan Estimator / Predictor (Logistic Regression)
# Inisialisasi model klasifikasi
model = LogisticRegression(random_state=42)

# Melatih model menggunakan data latih yang sudah diskalakan
model.fit(X_train_scaled, y_train)

# 4. Melakukan Prediksi dan Evaluasi
y_pred = model.predict(X_test_scaled)

# Menghitung akurasi model
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAkurasi Model Logistic Regression: {accuracy:.4f}")
print("\nLaporan Klasifikasi Lengkap:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

Dimensi X_train: (105, 4)
Dimensi X_test: (45, 4)

Akurasi Model Logistic Regression: 1.0000

Laporan Klasifikasi Lengkap:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        19
  versicolor       1.00      1.00      1.00        13
   virginica       1.00      1.00      1.00        13

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



## **5. Contoh Program Penerapan 2: Mencegah Data Leakage Menggunakan scikit-learn Pipeline**
Contoh kedua ini menunjukkan cara mengintegrasikan transformer dan estimator ke dalam objek Pipeline tunggal. Pendekatan ini merupakan standar industri (best practice) dalam pengembangan sistem machine learning profesional karena membuat kode lebih bersih, terstruktur, dan aman dari kesalahan pemrosesan data.

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Memuat dataset klasifikasi kanker payudara
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

# Membagi dataset menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# 2. Membangun Pipeline
# Pipeline menerima daftar tuple (nama_langkah, objek_transformer_atau_estimator)
# Langkah 1: Melakukan scaling data ('scaler')
# Langkah 2: Melakukan klasifikasi menggunakan Support Vector Classifier ('svc')
pipeline_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', C=1.0, random_state=42))
])

# 3. Melatih Pipeline secara utuh dengan satu perintah .fit()
# Di balik layar, scikit-learn akan otomatis memanggil fit_transform pada scaler,
# kemudian meneruskan hasilnya ke model SVC.
pipeline_model.fit(X_train, y_train)

# 4. Melakukan Evaluasi Langsung menggunakan metode .score() atau .predict()
# Saat .predict() dipanggil pada data uji, pipeline secara otomatis menerapkan
# transformasi scaling yang dipelajari dari data latih tanpa membocorkan data uji.
train_accuracy = pipeline_model.score(X_train, y_train)
test_accuracy = pipeline_model.score(X_test, y_test)

print(f"Akurasi pada Data Latih: {train_accuracy:.4f}")
print(f"Akurasi pada Data Uji: {test_accuracy:.4f}")

# Menampilkan confusion matrix dari hasil prediksi pipeline
y_pred = pipeline_model.predict(X_test)
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

Akurasi pada Data Latih: 0.9883
Akurasi pada Data Uji: 0.9720

Confusion Matrix:
[[52  2]
 [ 2 87]]
